In [55]:
from typing import TypeAlias

Clue: TypeAlias = list[int]
RowClues: TypeAlias = list[Clue]
ColClues: TypeAlias = list[Clue]
NonogramQuestion: TypeAlias = tuple[RowClues, ColClues]
NonogramState: TypeAlias = list[list[bool | None]]  # True: filled, False: empty, None: unknown

In [56]:
import tkinter as tk
import threading
import queue


class NonogramDrawer:
    def __init__(self) -> None:
        self.cell_size = 50
        self.padding = 5
        self._command_queue: queue.Queue = queue.Queue()
        self._ready = threading.Event()
        self._closed = threading.Event()
        self._thread: threading.Thread | None = None
        self.window: tk.Tk | None = None
        self.canvas: tk.Canvas | None = None

    def run(self) -> None:
        if self._thread is not None and self._thread.is_alive():
            return
        self._ready.clear()
        self._closed.clear()
        self._command_queue = queue.Queue()
        self._thread = threading.Thread(target=self._ui_thread, daemon=True, name="NonogramDrawerUI")
        self._thread.start()
        if not self._ready.wait(timeout=2):
            raise RuntimeError("Failed to start Nonogram drawer UI thread")

    def draw(self, question: NonogramQuestion, state: NonogramState | None = None) -> None:
        row_clues, col_clues = question
        rows = len(row_clues)
        cols = len(col_clues)

        if state is not None:
            if len(state) != rows or any(len(row) != cols for row in state):
                raise ValueError("State dimensions do not match the question grid size")

        self.run()
        if not self._ready.wait(timeout=2):
            raise RuntimeError("Drawer UI thread is not ready")
        self._command_queue.put(("draw", (question, state)))

    def close(self) -> None:
        if self._thread is None:
            return
        if not self._closed.is_set():
            self._command_queue.put(("close", None))
            self._closed.wait(timeout=2)
        if self._thread.is_alive():
            self._thread.join(timeout=2)
        self._thread = None
        self._ready.clear()

    def _ui_thread(self) -> None:
        self.window = tk.Tk()
        self.window.title("Nonogram Drawer")
        self.window.resizable(False, False)
        self.window.protocol("WM_DELETE_WINDOW", self._handle_close_request)
        self.canvas = None
        self._ready.set()
        self._closed.clear()
        self._poll_commands()
        self.window.mainloop()
        self._cleanup_after_loop()

    def _poll_commands(self) -> None:
        if self.window is None or self._closed.is_set():
            return
        try:
            while True:
                command, payload = self._command_queue.get_nowait()
                if command == "draw" and payload is not None:
                    question, state = payload
                    self._render(question, state)
                elif command == "close":
                    self._handle_close_request()
                else:
                    continue
        except queue.Empty:
            pass
        finally:
            if self.window is not None and not self._closed.is_set():
                self.window.after(16, self._poll_commands)

    def _handle_close_request(self) -> None:
        if self._closed.is_set():
            return
        self._closed.set()
        if self.window is not None:
            try:
                self.window.quit()
            except tk.TclError:
                pass
            try:
                self.window.destroy()
            except tk.TclError:
                pass

    def _cleanup_after_loop(self) -> None:
        self.canvas = None
        self.window = None
        self._closed.set()
        self._ready.clear()
        try:
            while True:
                self._command_queue.get_nowait()
        except queue.Empty:
            pass

    def _render(self, question: NonogramQuestion, state: NonogramState | None) -> None:
        if self.window is None:
            return
        row_clues, col_clues = question
        rows = len(row_clues)
        cols = len(col_clues)

        max_row_clues = max((len(c) for c in row_clues), default=0)
        max_col_clues = max((len(c) for c in col_clues), default=0)

        left_margin = max_row_clues * self.cell_size + self.padding * 2
        top_margin = max_col_clues * self.cell_size + self.padding * 2

        width = left_margin + cols * self.cell_size + self.padding
        height = top_margin + rows * self.cell_size + self.padding

        if self.canvas is None:
            self.canvas = tk.Canvas(self.window, bg="white")
            self.canvas.pack()

        self.canvas.configure(width=width, height=height)
        self.canvas.delete("all")

        # Draw row clues (right-aligned beside each row)
        for i, clues in enumerate(row_clues):
            y = top_margin + i * self.cell_size + self.cell_size / 2
            for k, v in enumerate(reversed(clues)):
                x = left_margin - (k + 0.5) * self.cell_size
                self.canvas.create_text(x, y, text=str(v), font=("Arial", int(self.cell_size * 0.4)))

        # Draw column clues (bottom-aligned above each column)
        for j, clues in enumerate(col_clues):
            x = left_margin + j * self.cell_size + self.cell_size / 2
            for k, v in enumerate(reversed(clues)):
                y = top_margin - (k + 0.5) * self.cell_size
                self.canvas.create_text(x, y, text=str(v), font=("Arial", int(self.cell_size * 0.4)))

        # Draw grid cells and state overlay if provided
        for i in range(rows):
            for j in range(cols):
                x0 = left_margin + j * self.cell_size
                y0 = top_margin + i * self.cell_size
                x1 = x0 + self.cell_size
                y1 = y0 + self.cell_size

                cell_state = state[i][j] if state is not None else None

                fill_color = "white"
                if cell_state is True:
                    fill_color = "black"
                elif cell_state is None:
                    fill_color = "white"  # Placeholder for unknown; adjust if you want a grey tone

                self.canvas.create_rectangle(x0, y0, x1, y1, outline="black", fill=fill_color)

                if cell_state is False:
                    inset = self.cell_size * 0.2
                    self.canvas.create_line(x0 + inset, y0 + inset, x1 - inset, y1 - inset, fill="gray40", width=2)
                    self.canvas.create_line(x0 + inset, y1 - inset, x1 - inset, y0 + inset, fill="gray40", width=2)


In [57]:
col_clues = [
    [0],
    [5],
    [2],
    [2],
    [0]
]
row_clues = [
    [3],
    [3],
    [1],
    [1],
    [1]
]
init_state: NonogramState = [[False, None, None, None, False] for _ in range(5)]

In [58]:
question: NonogramQuestion = (row_clues, col_clues)
# drawer: NonogramDrawer = NonogramDrawer()
# drawer.run()
# drawer.draw(question, init_state)

In [63]:
from copy import deepcopy
import time

class NonogramSolver:
    
    def __init__(self, question: NonogramQuestion, init_state: NonogramState | None = None) -> None:
        self.question = question
        if init_state is None:
            rows = len(question[0])
            cols = len(question[1])
            init_state = [[None for _ in range(cols)] for _ in range(rows)]
        self.state = deepcopy(init_state)
        
        self.drawer: NonogramDrawer = NonogramDrawer()
        self.drawer.run()
        self.drawer.draw(self.question, self.state)
        
    def handle_line_trivial(self, index: int, is_row: bool) -> list[tuple[int, int]]:
        """Handle trivial situations.

        Args:
            index (int): _description_
            is_row (bool): _description_
        Returns:
            list[tuple[int, int]]: List of modified coordinates (row, col)
        """
        # Placeholder for line handling logic
        if is_row:
            clues = self.question[0][index]
            line = self.state[index]
        else:
            clues = self.question[1][index]
            line = [self.state[i][index] for i in range(len(self.state))]
            
        if len(clues) > 1:
            return []
        elif True in line:
            return []
            
        max_clue = max(clues) if clues else 0
        # count max continuing none in line
        max_none = 0
        max_none_begin_idx = -1
        current_none = 0
        current_begin = 0
        for idx, cell in enumerate(line):
            if cell is None:
                if current_none == 0:
                    current_begin = idx
                current_none += 1
                if current_none > max_none:
                    max_none = current_none
                    max_none_begin_idx = current_begin
            else:
                current_none = 0
                
        # try to use specific experience rules
        if max_none < 2*max_clue <= 2*max_none and 0 <= max_none_begin_idx < len(line):
            # fill the middle part of the max none segment
            modified_coordinates: list[tuple[int, int]] = []
            start_fill = max_none_begin_idx + (max_none - max_clue)
            for fill_idx in range(start_fill, start_fill + max_clue):
                if is_row:
                    self.state[index][fill_idx] = True
                    modified_coordinates.append((index, fill_idx))
                else:
                    self.state[fill_idx][index] = True
                    modified_coordinates.append((fill_idx, index))
                    
            return modified_coordinates
        return []
                    
    def isLineValid(self, index: int, is_row: bool) -> bool:
        """验证指定行/列的当前状态是否符合线索（核心校验函数）
        规则：
        1. 对于任意一个已出现的连续 True 段，它的长度不能超过它“可能对应的任何一个 clue”的最大值；
        2. 连续块的数量不能超过线索的数量(?)；
        3. 若行/列已完全确定，需严格匹配线索的连续块长度和数量。
        """
        if is_row:
            clues = self.question[0][index]
            line = self.state[index]
        else:
            clues = self.question[1][index]
            line = [self.state[i][index] for i in range(len(self.state))]
        
        # Placeholder for line validity check logic
        blocks: Clue = []
        current_block_length = 0
        for cell in line:
            if cell is True:
                current_block_length += 1
            elif cell is False:
                if current_block_length > 0:
                    blocks.append(current_block_length)
                    current_block_length = 0
        if current_block_length > 0:
            blocks.append(current_block_length) # Append the last block if it exists
            
        if not blocks and clues == [0]:
            return True             # Special case: no blocks and clue is [0]
            
        # Check against clues
        # rule 2
        def count_min_blocks(line):
            exist_count = 0
            has_true = False
            for cell in line:
                if cell is False:
                    if has_true:
                        has_true = False
                        exist_count += 1
                elif cell is True:  
                    has_true = True
            if has_true:
                exist_count += 1
            return exist_count

        if count_min_blocks(line) > len(clues):
            print(f"break rule 2: current line: {line}, clues: {clues}, blocks: {blocks}")
            return False
        
        # rule 1
        max_clue = max(clues) if clues else 0
        for block_length in blocks:
            if block_length > max_clue:
                print(f"break rule 1: current line: {line}, clues: {clues}, blocks: {blocks}")
                return False
            
        # rule 3
        if None not in line:
            # return blocks == clues
            if blocks != clues:
                print(f"break rule 3: current line: {line}, clues: {clues}, blocks: {blocks}")
                return False
            else:
                return True
        
        # rule 4: 放得下
        # if sum(clues) + len(clues) - 1 > len(line):
        #     return False
        # if sum(clues) - line.count(True) > line.count(None):
        #     return False
        
        return True
        

    def isValid(self, question: NonogramQuestion, state: NonogramState) -> bool:
        # Placeholder for validity check logic
        for row_index in range(len(question[0])):
            if not self.isLineValid(row_index, is_row=True):
                print(f"Invalid row at index {row_index}")
                return False
        for col_index in range(len(question[1])):
            if not self.isLineValid(col_index, is_row=False):
                print(f"Invalid column at index {col_index}")
                return False
        return True
    
    def findFirstUnknown(self) -> tuple[int, int] | None:
        for i in range(len(self.state)):
            for j in range(len(self.state[0])):
                if self.state[i][j] is None:
                    return (i, j)
        return None
    
    def solve(self) -> NonogramState | None:
        self.drawer.draw(self.question, self.state)
        
        if not self.isValid(self.question, self.state):
            print("Initial state invalid")
            return None
        
        set_true_positions: list[tuple[int, int]] = []
        for row_idx in range(len(self.state)):
            set_true_positions.extend(self.handle_line_trivial(row_idx, is_row=True))
        for col_idx in range(len(self.state[0])):
            set_true_positions.extend(self.handle_line_trivial(col_idx, is_row=False))
            
        return self._solve()
        
    
    def _solve(self) -> NonogramState | None:
        
        self.drawer.draw(self.question, self.state)
        print("_solve")
        
        if not self.isValid(self.question, self.state):
            return None
        
        unknown_pos = self.findFirstUnknown()
        if unknown_pos is None:
            return self.state
        
        row, col = unknown_pos
        for value in [True, False]:
            # Try setting the cell to the current value
            self.state[row][col] = value
            
            result = self._solve()
            if result is not None:
                return result
            
            # Backtrack
            self.state[row][col] = None
                
        return None
        

In [64]:
solver: NonogramSolver = NonogramSolver(question, init_state)
result = solver.solve()
if result is not None:
    print("Solved Nonogram:")
    solver.drawer.draw(solver.question, result)
else:
    print("No solution found.")

_solve
_solve
_solve
break rule 1: current line: [False, True, True, None, False], clues: [1], blocks: [2]
Invalid row at index 2
_solve
_solve
break rule 2: current line: [False, True, False, True, False], clues: [1], blocks: [1, 1]
Invalid row at index 2
_solve
_solve
_solve
break rule 1: current line: [False, True, True, None, False], clues: [1], blocks: [2]
Invalid row at index 3
_solve
_solve
break rule 2: current line: [False, True, False, True, False], clues: [1], blocks: [1, 1]
Invalid row at index 3
_solve
_solve
_solve
break rule 1: current line: [False, True, True, None, False], clues: [1], blocks: [2]
Invalid row at index 4
_solve
_solve
break rule 2: current line: [False, True, False, True, False], clues: [1], blocks: [1, 1]
Invalid row at index 4
_solve
Solved Nonogram:
